In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import html
import base64
import uuid

# ============================================================
#  1. CARGA DE DATOS (DESDE ZIP EN TU GITHUB)
# ============================================================
zip_url = 'https://raw.githubusercontent.com/Yersitoo/tp-octubre/main/winemag-data-130k-v2.zip'
print("1/5 - Descargando y descomprimiendo el dataset desde tu GitHub...")

try:
    df = pd.read_csv(zip_url, compression='zip')
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
except Exception as e:
    raise RuntimeError(f"❌ Error al cargar el dataset: {e}")

# ============================================================
#  2. MAPA DE BURBUJAS — ITALIA Y FRANCIA (Sin renderizar en pantalla)
# ============================================================
print("2/5 - Generando mapas de Europa...")
df_euro = df[df['country'].isin(['France', 'Italy'])].copy()

coordenadas_vino = {
    'Alsace': (48.3182, 7.4416), 'Beaujolais': (46.1500, 4.6500), 'Bordeaux': (44.8378, -0.5792),
    'Burgundy': (47.0525, 4.3837), 'Champagne': (48.9566, 4.3676), 'France Other': (46.6034, 1.8883),
    'Languedoc-Roussillon': (43.4000, 3.1000), 'Loire Valley': (47.3941, 0.6848), 'Provence': (43.9352, 6.0679),
    'Rhône Valley': (44.9333, 4.8924), 'Southwest France': (43.6047, 1.4442),
    'Central Italy': (42.5000, 12.5000), 'Italy Other': (41.9028, 12.4964), 'Lombardy': (45.4642, 9.1900),
    'Northeastern Italy': (46.0000, 12.0000), 'Northwestern Italy': (45.5000, 8.5000), 'Piedmont': (45.0500, 7.6667),
    'Sicily & Sardinia': (37.5990, 14.0154), 'Southern Italy': (40.5000, 16.0000), 'Tuscany': (43.7695, 11.2558),
    'Veneto': (45.4343, 12.3388),
}

df_euro['lat'] = df_euro['province'].map(lambda x: coordenadas_vino.get(x, (None, None))[0])
df_euro['lon'] = df_euro['province'].map(lambda x: coordenadas_vino.get(x, (None, None))[1])
df_euro = df_euro.dropna(subset=['lat', 'lon'])
euro_data = df_euro.groupby(['province', 'lat', 'lon', 'country']).agg(promedio_puntos=('points', 'mean'), cantidad_vinos=('points', 'count')).reset_index()

france_data = euro_data[euro_data['country'] == 'France'].copy()
italy_data  = euro_data[euro_data['country'] == 'Italy'].copy()

COLORSCALE = [[0.0, '#f7c5c5'], [0.4, '#c22d41'], [0.75, '#8a052b'], [1.0, '#d4af37']]
def normalize_sizes(series, min_px=12, max_px=60):
    s_min, s_max = series.min(), series.max()
    if s_max == s_min: return pd.Series([max_px / 2] * len(series))
    return min_px + (series - s_min) / (s_max - s_min) * (max_px - min_px)

def build_scatter(data, colorscale, label_country):
    sizes = normalize_sizes(data['cantidad_vinos'])
    scores = data['promedio_puntos']
    hover_text = [f"<b>{row.province}</b><br>🍷 Vinos: <b>{int(row.cantidad_vinos):,}</b><br>⭐ Puntuación media: <b>{row.promedio_puntos:.1f}</b>" for _, row in data.iterrows()]
    return go.Scattergeo(lat=data['lat'], lon=data['lon'], mode='markers+text', name=label_country,
        marker=dict(size=sizes, color=scores, colorscale=colorscale, cmin=87, cmax=92, opacity=0.88, line=dict(width=1.2, color='rgba(43,0,11,0.35)'),
                    colorbar=dict(title=dict(text="Puntuación<br>media", font=dict(color="#2b000b", size=12, family="Georgia, serif"), side='right'),
                                  tickfont=dict(color="#7a4510", size=11), thickness=14, len=0.60, bgcolor='rgba(253,245,236,0.8)',
                                  bordercolor='rgba(158,126,110,0.4)', borderwidth=1, tickmode='linear', tick0=87, dtick=1, x=1.02), showscale=True),
        text=data['province'], textposition='top center', textfont=dict(size=10, color="#2b000b", family="Georgia, serif"),
        hovertext=hover_text, hoverinfo='text', hoverlabel=dict(bgcolor='#ffffff', bordercolor='rgba(43,0,11,0.3)', font=dict(color='#2b000b', size=13, family="Georgia, serif")))

def build_figure(data, country_name, lon_range, lat_range, title_str):
    fig = go.Figure(build_scatter(data, COLORSCALE, country_name))
    fig.update_geos(scope='europe', showland=True, landcolor="#ead9c2", showocean=True, oceancolor="#c8dce8", showlakes=True, lakecolor="#c8dce8", showrivers=True, rivercolor='#1e3a5f', showcoastlines=True, coastlinecolor="#9e7e6e", coastlinewidth=0.8, showframe=False, showcountries=True, countrycolor="#9e7e6e", countrywidth=0.8, showsubunits=True, subunitcolor='rgba(100,120,150,0.4)', subunitwidth=0.5, lonaxis=dict(range=lon_range, showgrid=True, gridcolor="#e0c8b0", gridwidth=0.3), lataxis=dict(range=lat_range, showgrid=True, gridcolor="#e0c8b0", gridwidth=0.3), bgcolor="#fdf5ec", resolution=50)
    fig.update_layout(title=dict(text=title_str, x=0.5, xanchor='center', font=dict(size=22, color="#2b000b", family="Georgia, serif"), pad=dict(b=10)), paper_bgcolor="#fdf5ec", geo=dict(bgcolor="#fdf5ec"), margin=dict(l=10, r=100, t=70, b=30), width=900, height=620, annotations=[dict(text=(f"<b>{len(data)}</b> regiones · <b>{int(data['cantidad_vinos'].sum()):,}</b> vinos totales · Puntuación media: <b>{data['promedio_puntos'].mean():.1f}</b>"), xref='paper', yref='paper', x=0.5, y=-0.04, showarrow=False, font=dict(size=11, color="#7a4510", family="Georgia, serif"), xanchor='center')])
    return fig

fig_france = build_figure(france_data, 'Francia', [-5.5, 10.0], [41.5, 51.5], "🍷 <b>Regiones Vitivinícolas de Francia</b><br><sup style='color:#a0aec0'>Volumen (tamaño) · Puntuación media (color)</sup>")
fig_italy = build_figure(italy_data, 'Italia', [6.5, 18.5], [36.5, 47.5], "🍷 <b>Regiones Vitivinícolas de Italia</b><br><sup style='color:#a0aec0'>Volumen (tamaño) · Puntuación media (color)</sup>")

# ============================================================
#  3. MAPA COROPLÉTICO DE ESTADOS UNIDOS
# ============================================================
print("3/5 - Generando mapa de Estados Unidos...")
df_us = df[df['country'] == 'US'].copy()
state_codes = {'California': 'CA', 'Washington': 'WA', 'Oregon': 'OR', 'New York': 'NY', 'Virginia': 'VA', 'Michigan': 'MI', 'Idaho': 'ID', 'Texas': 'TX', 'Pennsylvania': 'PA', 'New Jersey': 'NJ', 'Missouri': 'MO', 'Colorado': 'CO', 'New Mexico': 'NM', 'Massachusetts': 'MA', 'Ohio': 'OH', 'Illinois': 'IL', 'Nevada': 'NV', 'Maryland': 'MD', 'North Carolina': 'NC', 'Arizona': 'AZ'}
df_us['state_code'] = df_us['province'].map(state_codes)
df_us = df_us.dropna(subset=['state_code'])

us_data = df_us.groupby('state_code').agg(promedio_puntos=('points', 'mean'), cantidad_vinos=('points', 'count')).reset_index()
us_data['hover'] = "<b>" + us_data['state_code'] + "</b><br>🍷 Vinos: <b>" + us_data['cantidad_vinos'].apply(lambda x: f"{x:,}") + "</b><br>⭐ Puntuación: <b>" + us_data['promedio_puntos'].apply(lambda x: f"{x:.1f}") + "</b>"

fig_us = go.Figure(go.Choropleth(locations=us_data['state_code'], locationmode="USA-states", z=us_data['promedio_puntos'], colorscale=COLORSCALE, zmin=87, zmax=92, hovertext=us_data['hover'], hoverinfo='text', hoverlabel=dict(bgcolor='#fdf5ec', bordercolor='rgba(158,126,110,0.6)', font=dict(color="#2b000b", size=12, family="Georgia, serif")), marker_line_color='rgba(158,126,110,0.5)', marker_line_width=1.0, colorbar=dict(title=dict(text="Puntuación", font=dict(color="#2b000b", size=12, family="Georgia, serif"), side='right'), tickfont=dict(color="#7a4510", size=11), thickness=14, len=0.60, bgcolor='rgba(253,245,236,0.85)', bordercolor='rgba(158,126,110,0.4)', borderwidth=1, tickmode='linear', tick0=87, dtick=1, x=1.02)))
fig_us.update_layout(title=dict(text=("🍷 <b>Calidad Vitivinícola por Estado — EE. UU.</b><br><sup style='color:#d4af37'>Puntuación media</sup>"), x=0.5, xanchor='center', font=dict(size=21, color="#2b000b", family="Georgia, serif")), geo=dict(scope='usa', bgcolor="#fdf5ec", landcolor='#ead9c2', lakecolor='#c8dce8', showlakes=True, showcoastlines=True, coastlinecolor='rgba(158,126,110,0.5)', coastlinewidth=0.8, showframe=False, showcountries=False), paper_bgcolor="#fdf5ec", margin=dict(l=10, r=110, t=80, b=30), width=900, height=580)

# ============================================================
#  4. GRÁFICOS ANALÍTICOS Y PERFIL SENSORIAL
# ============================================================
print("4/5 - Generando gráficos analíticos...")
df_modelo = df.copy()

if 'price_tier' not in df_modelo.columns and 'price' in df_modelo.columns:
    df_modelo['price_tier'] = df_modelo['price'].apply(lambda p: "Sin precio informado" if pd.isna(p) else ("Económico" if p < 20 else ("Gama Media" if p <= 50 else "Premium")))
else:
    df_modelo['price_tier'] = df_modelo['price_tier'].replace({"1. Económico (<$20)": "Económico", "2. Gama Media ($20-$50)": "Gama Media", "3. Premium (>$50)": "Premium"})

orden_precios = [p for p in ["Económico", "Gama Media", "Premium", "Sin precio informado"] if p in df_modelo['price_tier'].unique()]
labels_esp = {"price_tier": "Segmento Comercial", "points": "Puntaje de Calidad", "price": "Precio Promedio (USD)", "Cantidad": "Cantidad de Vinos", "Sabor": "Perfil Aromático"}

sabores_buscar = ['oak', 'tannin', 'cherry', 'plum', 'berry', 'vanilla', 'spice', 'fruit', 'acid', 'dry']
df_desc = df_modelo['description'].fillna('').str.lower()
dummies_sensorial = pd.DataFrame()
for sabor in sabores_buscar:
    sabor_es = {'oak': 'Roble', 'tannin': 'Taninos', 'cherry': 'Cereza', 'plum': 'Ciruela', 'berry': 'Baya', 'vanilla': 'Vainilla', 'spice': 'Especias', 'fruit': 'Fruta', 'acid': 'Acidez', 'dry': 'Seco'}.get(sabor, sabor.capitalize())
    dummies_sensorial[sabor_es] = df_desc.str.contains(sabor).astype(int)

fig1 = px.box(df_modelo, x="price_tier", y="points", color="price_tier", category_orders={"price_tier": orden_precios}, labels=labels_esp, color_discrete_sequence=['#ff9999', '#c22d41', '#5c001e', '#b31237'])
fig1.update_layout(showlegend=False, template="plotly_white", margin=dict(l=50, r=20, t=20, b=80))

top_sabores = dummies_sensorial.sum().sort_values(ascending=False).head(10)
df_sabores = pd.DataFrame({'Sabor': top_sabores.index, 'Cantidad': top_sabores.values})
fig2 = px.bar(df_sabores, x='Cantidad', y='Sabor', orientation='h', labels=labels_esp, color='Cantidad', color_continuous_scale=['#ff9999', '#c22d41', '#5c001e'])
fig2.update_layout(yaxis={'categoryorder':'total ascending'}, template="plotly_white", margin=dict(l=100, r=20, t=20, b=60), coloraxis_showscale=False)

df_scatter = df_modelo[df_modelo['price'] <= 150]
fig3 = px.scatter(df_scatter, x="price", y="points", color="price_tier", category_orders={"price_tier": orden_precios}, labels=labels_esp, color_discrete_sequence=['#ff9999', '#c22d41', '#5c001e', '#b31237'])
fig3.update_layout(template="plotly_white", margin=dict(l=50, r=20, t=20, b=120), legend=dict(title="Segmento de Mercado:", orientation="h", yanchor="top", y=-0.25, xanchor="center", x=0.5))

# ============================================================
#  5. GENERAR EXPORTACIÓN A HTML FINAL
# ============================================================
print("5/5 - Empaquetando todo en el archivo HTML (esto puede tardar unos segundos)...")
def plot_to_b64(fig):
    html_bytes = fig.to_html(full_html=True, include_plotlyjs='cdn', default_height="100%", default_width="100%").encode('utf-8')
    return "data:text/html;base64," + base64.b64encode(html_bytes).decode('utf-8')

src1 = html.escape(fig1.to_html(full_html=True, include_plotlyjs='cdn', default_height="100%", default_width="100%"))
src2 = plot_to_b64(fig2)
src3 = plot_to_b64(fig3)
src4 = plot_to_b64(fig_france)
src5 = plot_to_b64(fig_italy)
src6 = plot_to_b64(fig_us)

_html_template = f'''
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>Data Winery - Pipeline ETL</title>
</head>
<body>
<div class="gx" id="__ID__" tabindex="0">
<style>
.gx {{ --vino: #5c001e; --vino-oscuro: #2b000b; --dorado: #d4af37; --papel: #fcf9f2; --texto: #333; font-family: 'Segoe UI', system-ui, sans-serif; background: var(--papel); color: var(--texto); border-radius: 8px; padding: 0; display: flex; flex-direction: column; box-sizing: border-box; outline: none; width: 100%; max-width: 100%; height: 93vh; overflow-y: auto; overflow-x: hidden; position: relative; }}
.gx * {{ box-sizing: border-box; }}
.gx.fs, .gx:fullscreen {{ width: 100vw; height: 100vh; max-width: none; border-radius: 0; overflow: auto; }}
.gx::-webkit-scrollbar {{ width: 10px; }}
.gx::-webkit-scrollbar-track {{ background: var(--papel); }}
.gx::-webkit-scrollbar-thumb {{ background: var(--vino); border-radius: 5px; }}
.gx .sticky-wrapper {{ position: sticky; top: 15px; width: 100%; height: 0; z-index: 100000; display: flex; justify-content: flex-end; padding-right: 15px; pointer-events: none; }}
.gx .btn-fs-mini {{ background: rgba(212,175,55,0.25); color: var(--dorado); border: 1px solid var(--dorado); border-radius: 6px; font-weight: bold; cursor: pointer; transition: all 0.3s ease; pointer-events: auto; width: 42px; height: 42px; display: flex; align-items: center; justify-content: flex-start; padding: 0; box-shadow: 0 4px 10px rgba(0,0,0,0.4); backdrop-filter: blur(3px); overflow: hidden; }}
.gx .fs-icon {{ min-width: 40px; display: flex; align-items: center; justify-content: center; font-size: 1.4em; }}
.gx .fs-text {{ white-space: nowrap; font-size: 0.95em; opacity: 0; transition: opacity 0.3s ease; margin-left: 5px; }}
.gx .btn-fs-mini:hover {{ width: 200px; background: var(--dorado); color: var(--vino-oscuro); }}
.gx .btn-fs-mini:hover .fs-text {{ opacity: 1; }}
.gx #__ID__-intro {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; background-color: var(--vino-oscuro); z-index: 9999; display: flex; flex-direction: column; align-items: center; justify-content: center; transition: opacity 1.5s ease; overflow: hidden; text-align: center; padding: 20px; }}
.gx .intro-bg {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; z-index: 0; background: #000; }}
.gx .intro-bg img, .gx .carousel-bg img {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; object-fit: cover; opacity: 0; transition: opacity 1.2s ease-in-out; }}
.gx .intro-bg img.active, .gx .carousel-bg img.active {{ opacity: 1; }}
.gx .intro-overlay, .gx .hero-overlay {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; background: linear-gradient(rgba(43,0,11,0.85), rgba(92,0,30,0.85)); z-index: 1; }}
.gx .btn-comenzar {{ background-color: var(--dorado); color: var(--vino-oscuro); border: none; position: relative; z-index: 5; padding: 18px 55px; font-size: 1.6em; font-weight: bold; border-radius: 40px; cursor: pointer; box-shadow: 0 10px 25px rgba(0,0,0,0.6); transition: all 0.3s ease; letter-spacing: 2px; text-transform: uppercase; font-family: 'Segoe UI', sans-serif; margin-bottom: 20px; }}
.gx .btn-comenzar:hover {{ background-color: #fff; transform: scale(1.06); }}
.gx #__ID__-introContent {{ display: none; flex-direction: column; align-items: center; justify-content: center; width: 100%; height: 100%; z-index: 4; position: relative; }}
.gx #__ID__-memberStack {{ display: flex; flex-direction: column; align-items: center; gap: 18px; transition: transform 0.6s cubic-bezier(.55,0,.85,.35), opacity 0.5s ease; transform-origin: center center; position: absolute; }}
.gx #__ID__-memberStack.crushed {{ transform: scaleY(0.02); opacity: 0; }}
.gx .member {{ font-family: 'Georgia', serif; font-size: 2.8em; color: #f1ece1; text-shadow: 0 4px 10px rgba(0,0,0,0.5); opacity: 0; transform: translateY(20px); transition: opacity 0.5s ease, transform 0.5s cubic-bezier(.2,.8,.2,1); }}
.gx .member .ln {{ color: var(--dorado); font-weight: bold; }}
.gx #__ID__-trubo-presenta {{ position: absolute; font-family: 'Georgia', serif; font-size: 4.5em; color: var(--dorado); font-style: italic; opacity: 0; transition: opacity 1s ease; text-shadow: 0 10px 30px rgba(0,0,0,0.9); display: none; }}
.gx #__ID__-introTitle {{ position: absolute; text-align: center; width: 90%; opacity: 0; transform: scale(0.9); transition: all 1.5s cubic-bezier(.2,0.8,.2,1); display: none; }}
.gx #__ID__-introTitle.show {{ opacity: 1; transform: scale(1); }}
.gx #__ID__-introTitle h1 {{ font-family: 'Georgia', serif; font-size: 5.5em; color: var(--dorado); margin: 0; text-shadow: 0 15px 40px rgba(0,0,0,0.9); letter-spacing: 4px; line-height: 1.1; text-transform: uppercase; }}
.gx #__ID__-introTitle .sub {{ font-size: 1.8em; color: #fcf9f2; margin-top: 15px; font-style: italic; opacity: 0.9; letter-spacing: 1px; }}
.gx #__ID__-main {{ display: none; opacity: 0; transition: opacity 1s ease; width: 100%; height: 100%; }}
.gx .hero {{ position: relative; height: 280px; width: 100%; display: flex; align-items: center; justify-content: center; overflow: hidden; flex-shrink: 0; }}
.gx .carousel-bg {{ position: absolute; top: 0; left: 0; width: 100%; height: 100%; z-index: 0; background: #000; }}
.gx .hero-content {{ position: relative; z-index: 2; text-align: center; color: white; width: 100%; padding: 0 20px; }}
.gx .hero-content h1 {{ font-size: 3.2em; color: var(--dorado); margin: 0 0 10px 0; letter-spacing: 2px; text-transform: uppercase; font-family: 'Georgia', serif; }}
.gx .hero-content p {{ font-size: 1.2em; font-style: italic; margin: 0; opacity: 0.9; }}
.gx .tabs {{ display: flex; width: 100%; background: var(--vino-oscuro); position: sticky; top: 0; z-index: 10; box-shadow: 0 4px 6px rgba(0,0,0,0.2); }}
.gx .tabs button {{ flex: 1; background: none; border: none; color: white; padding: 16px; font-size: 1.05em; cursor: pointer; transition: 0.3s; border-bottom: 4px solid transparent; text-transform: uppercase; letter-spacing: 1px; }}
.gx .tabs button:hover {{ background: rgba(255,255,255,0.1); }}
.gx .tabs button.on {{ border-bottom-color: var(--dorado); color: var(--dorado); font-weight: bold; background: rgba(0,0,0,0.3); }}
.gx .tab-content {{ display: none; padding: 30px 3% 50px 3%; animation: gxin 0.4s; width: 100%; }}
.gx .tab-content.active {{ display: block; }}
.gx h2.seccion {{ color: var(--vino); border-bottom: 2px solid var(--dorado); padding-bottom: 8px; margin-top: 0; font-size: 1.8em; font-family: 'Georgia', serif; }}
.gx .graph-gallery {{ position: relative; width: 100%; display: flex; flex-direction: column; align-items: center; margin-top: 15px; padding: 0 20px; }}
.gx .gallery-inner {{ position: relative; width: 100%; max-width: 960px; display: flex; align-items: center; justify-content: center; }}
.gx .gallery-btn {{ position: absolute; top: 45%; transform: translateY(-50%); background: rgba(43,0,11,0.9); color: var(--dorado); border: 2px solid var(--dorado); width: 45px; height: 45px; border-radius: 50%; font-size: 1.4em; display: flex; align-items: center; justify-content: center; cursor: pointer; z-index: 20; transition: 0.3s; box-shadow: 0 4px 10px rgba(0,0,0,0.3); }}
.gx .gallery-btn:hover {{ background: var(--dorado); color: var(--vino-oscuro); transform: translateY(-50%) scale(1.1); }}
.gx .gallery-btn.prev {{ left: -25px; }}
.gx .gallery-btn.next {{ right: -25px; }}
.gx .graph-slide {{ display: none; width: 100%; flex-direction: column; align-items: center; animation: gxin 0.5s; }}
.gx .graph-slide.active {{ display: flex; }}
.gx .slide-card-container {{ background: white; padding: 28px 34px; border-radius: 8px; box-shadow: 0 6px 15px rgba(0,0,0,0.08); width: 100%; min-height: 420px; display: flex; flex-direction: column; justify-content: flex-start; border-left: 6px solid var(--dorado); margin-bottom: 15px; overflow-y: auto; max-height: 490px; }}
.gx .slide-card-container h3 {{ color: var(--vino); margin-top: 0; margin-bottom: 8px; font-size: 1.65em; font-family: 'Georgia', serif; }}
.gx .slide-card-container p {{ font-size: 1.02em; color: #444; line-height: 1.65; margin-bottom: 6px; }}
.gx .slide-card-container code {{ background: rgba(92,0,30,0.08); color: var(--vino); padding: 1px 5px; border-radius: 3px; font-family: 'Consolas', monospace; font-size: 0.9em; }}
.gx .slide-card-container strong {{ color: #2b000b; }}
.gx .slide-card-container em {{ color: #5c001e; font-style: italic; }}
.gx .code-label {{ display: flex; align-items: center; gap: 7px; font-size: 0.70em; font-weight: 700; letter-spacing: 1.3px; text-transform: uppercase; color: var(--vino); margin-top: 16px; margin-bottom: 4px; font-family: 'Segoe UI', sans-serif; }}
.gx .code-label::before {{ content: ''; display: inline-block; width: 7px; height: 7px; border-radius: 50%; background: var(--dorado); flex-shrink: 0; }}
.gx .code-block {{ background: #1a0008; border: 1px solid rgba(212,175,55,0.3); border-left: 3px solid var(--dorado); border-radius: 6px; padding: 13px 16px; font-family: 'Consolas', 'Courier New', monospace; font-size: 0.82em; color: #f1ece1; overflow-x: auto; white-space: pre; line-height: 1.75; box-shadow: inset 0 1px 8px rgba(0,0,0,0.5); }}
.gx .code-block .kw  {{ color: #ff9999; }}
.gx .code-block .fn  {{ color: #d4af37; }}
.gx .code-block .str {{ color: #fce8b2; }}
.gx .code-block .cm  {{ color: rgba(212,175,55,0.55); font-style: italic; }}
.gx .code-block .num {{ color: #e8c97a; }}
.gx .code-block .op  {{ color: #ff9999; }}
.gx .code-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 14px; width: 100%; }}
.gx .code-grid-col {{ display: flex; flex-direction: column; }}
.gx .grafico-box.grande {{ background: white; padding: 10px; border-radius: 8px; box-shadow: 0 6px 15px rgba(0,0,0,0.1); width: 100%; height: 530px; overflow: hidden; margin-bottom: 15px; display: flex; justify-content: center; align-items: center; }}
.gx .grafico-box.grande iframe {{ width: 100%; height: 100%; border: none; display: block; }}
.gx .graph-desc {{ background: #fff; padding: 18px; border-radius: 8px; border-left: 5px solid var(--dorado); width: 100%; box-shadow: 0 4px 10px rgba(0,0,0,0.05); }}
.gx .graph-desc h3 {{ color: var(--vino); margin-top: 0; font-size: 1.25em; }}
.gx .graph-desc p {{ margin: 0; font-size: 1.02em; color: #555; line-height: 1.5; }}
.gx .pagination-dots {{ display: flex; gap: 8px; align-items: center; justify-content: center; margin-top: 15px; }}
.gx .dot {{ width: 10px; height: 10px; border-radius: 50%; background-color: #d1c2c7; transition: all 0.3s ease; }}
.gx .dot.active {{ width: 28px; border-radius: 6px; background-color: var(--vino); }}
@keyframes gxin {{ from {{ opacity: 0; transform: translateY(15px); }} to {{ opacity: 1; transform: none; }} }}
</style>

<div class="sticky-wrapper">
    <button class="btn-fs-mini" id="__ID__-fs">
        <span class="fs-icon"><svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5"><path d="M8 3H5a2 2 0 0 0-2 2v3m18 0V5a2 2 0 0 0-2-2h-3m0 18h3a2 2 0 0 0 2-2v-3M3 16v3a2 2 0 0 0 2 2h3"></path></svg></span>
        <span class="fs-text" id="__ID__-fs-text">Maximizar pantalla</span>
    </button>
</div>

<div id="__ID__-intro">
    <div class="intro-bg" id="__ID__-intro-carousel"></div>
    <div class="intro-overlay"></div>
    <div id="__ID__-btnBox" style="z-index:5;">
        <button class="btn-comenzar" onclick="iniciarPresentacion__ID__()">Comenzar</button>
    </div>
    <div id="__ID__-introContent">
        <div id="__ID__-memberStack">
            <div class="member"><span class="ln">CAMPOS</span>, Fiorella</div>
            <div class="member"><span class="ln">MARECO</span>, Carlos</div>
            <div class="member"><span class="ln">SALVATORE</span>, Santiago</div>
            <div class="member"><span class="ln">SITTNER</span>, Germán</div>
        </div>
        <div id="__ID__-trubo-presenta">Grupo Turbo presenta...</div>
        <div id="__ID__-introTitle">
            <h1>ETL &amp; ANÁLISIS CON<br><span style="color:#fff;">OpenRefine</span></h1>
            <div class="sub">Pipeline de Transformación y Modelado sobre Datos Masivos</div>
        </div>
    </div>
</div>

<div id="__ID__-main">
    <div class="hero">
        <div class="carousel-bg" id="__ID__-main-carousel"></div>
        <div class="hero-overlay"></div>
        <div class="hero-content">
            <h1>ETL &amp; OpenRefine</h1>
            <p>Procesamiento, Limpieza y Estandarización de Datos</p>
        </div>
    </div>

    <div class="tabs" id="__ID__-tabs">
        <button class="on" onclick="openTab__ID__(0)">1. La Tecnología</button>
        <button onclick="openTab__ID__(1)">2. El Dataset</button>
        <button onclick="openTab__ID__(2)">3. Análisis Visual</button>
    </div>

    <!-- PESTAÑA 1 -->
    <div class="tab-content active" id="__ID__-c0">
        <h2 class="seccion">Arquitectura Tecnológica</h2>
        <div class="graph-gallery">
            <div class="gallery-inner">
                <button class="gallery-btn prev" onclick="changeTechSlide__ID__(-1)">❮</button>
                <div class="graph-slide active tech-slide">
                    <div class="slide-card-container" style="font-size: 1.15em; line-height: 1.8;">
                        <h3 style="font-size: 1.9em; margin-bottom: 15px;">OpenRefine: Motor de Auditoría y ETL</h3>
                        <p>Herramienta <i>open-source</i> especializada en la depuración masiva de datos semiestructurados y heterogéneos. A diferencia de las hojas de cálculo tradicionales, OpenRefine opera mediante una arquitectura de servidor local orientada a bases de datos, lo que le permite procesar cientos de miles de registros sin bloqueos ni pérdida de rendimiento.</p>
                    </div>
                </div>
                <div class="graph-slide tech-slide">
                    <div class="slide-card-container" style="font-size: 1.15em; line-height: 1.8;">
                        <h3 style="font-size: 1.9em; margin-bottom: 15px;">GREL: General Refine Expression Language</h3>
                        <p>GREL es el poderoso lenguaje funcional nativo de OpenRefine, diseñado específicamente para la manipulación avanzada de cadenas de texto y el enriquecimiento de datos.</p>
                    </div>
                </div>
                <div class="graph-slide tech-slide">
                    <div class="slide-card-container" style="font-size: 1.15em; line-height: 1.8;">
                        <h3 style="font-size: 1.9em; margin-bottom: 15px;">Clustering y Repositorio Reproducible</h3>
                        <p>La normalización de nombres comerciales provenientes de más de 12 países y múltiples idiomas es un desafío clásico de <i>Data Quality</i>.</p>
                    </div>
                </div>
                <button class="gallery-btn next" onclick="changeTechSlide__ID__(1)">❯</button>
            </div>
            <div class="pagination-dots" id="tech-dots-__ID__"><span class="dot active"></span><span class="dot"></span><span class="dot"></span></div>
        </div>
    </div>

    <!-- PESTAÑA 2 -->
    <div class="tab-content" id="__ID__-c1">
        <h2 class="seccion">Pipeline de Limpieza y Enriquecimiento</h2>
        <div class="graph-gallery">
            <div class="gallery-inner">
                <button class="gallery-btn prev" onclick="changeDataSlide__ID__(-1)">❮</button>
                <div class="graph-slide active data-slide">
                    <div class="slide-card-container">
                        <h3>1. Depuración Estructural y Tipado</h3>
                        <div class="code-grid">
                            <div class="code-grid-col"><div class="code-label">OpenRefine · Eliminar columna</div><div class="code-block"><span class="cm">// core/column-removal</span></div></div>
                            <div class="code-grid-col"><div class="code-label">GREL · Conversión a numérico</div><div class="code-block"><span class="fn">value</span>.<span class="kw">toNumber</span>()</div></div>
                        </div>
                    </div>
                </div>
                <div class="graph-slide data-slide">
                    <div class="slide-card-container">
                        <h3>2. Extracción de Añadas (Regex)</h3>
                        <div class="code-label">GREL · Nueva columna vintage_year</div>
                        <div class="code-block"><span class="fn">grel:</span><span class="kw">with</span>(value.<span class="kw">match</span>(<span class="str">/.*?\\\\b(19\\\\d\\\\d|20[0-2]\\\\d)\\\\b.*/</span>), m, <span class="kw">if</span>(<span class="fn">isNotNull</span>(m), m[<span class="num">0</span>].<span class="kw">toNumber</span>(), <span class="kw">null</span>))</div>
                    </div>
                </div>
                <div class="graph-slide data-slide">
                    <div class="slide-card-container">
                        <h3>3. Clustering y Segmentación Comercial</h3>
                    </div>
                </div>
                <div class="graph-slide data-slide">
                    <div class="slide-card-container">
                        <h3>4. Enriquecimiento Sensorial y API REST</h3>
                    </div>
                </div>
                <div class="graph-slide data-slide">
                    <div class="slide-card-container">
                        <h3>5. Ingeniería de Variables</h3>
                    </div>
                </div>
                <button class="gallery-btn next" onclick="changeDataSlide__ID__(1)">❯</button>
            </div>
            <div class="pagination-dots" id="data-dots-__ID__"><span class="dot active"></span><span class="dot"></span><span class="dot"></span><span class="dot"></span><span class="dot"></span></div>
        </div>
    </div>

    <!-- PESTAÑA 3 -->
    <div class="tab-content" id="__ID__-c2">
        <h2 class="seccion">Exploración Interactiva de Datos</h2>
        <div class="graph-gallery">
            <div class="gallery-inner">
                <button class="gallery-btn prev" onclick="changeGraphSlide__ID__(-1)">❮</button>
                <div class="graph-slide active graph-slide-item"><div class="grafico-box grande"><iframe srcdoc="{src1}" scrolling="no"></iframe></div></div>
                <div class="graph-slide graph-slide-item"><div class="grafico-box grande"><iframe src="{src2}" scrolling="no"></iframe></div></div>
                <div class="graph-slide graph-slide-item"><div class="grafico-box grande"><iframe src="{src3}" scrolling="no"></iframe></div></div>
                <div class="graph-slide graph-slide-item"><div class="grafico-box grande"><iframe src="{src4}" scrolling="no"></iframe></div></div>
                <div class="graph-slide graph-slide-item"><div class="grafico-box grande"><iframe src="{src5}" scrolling="no"></iframe></div></div>
                <div class="graph-slide graph-slide-item"><div class="grafico-box grande"><iframe src="{src6}" scrolling="no"></iframe></div></div>
                <button class="gallery-btn next" onclick="changeGraphSlide__ID__(1)">❯</button>
            </div>
            <div class="pagination-dots" id="graph-dots-__ID__"><span class="dot active"></span><span class="dot"></span><span class="dot"></span><span class="dot"></span><span class="dot"></span><span class="dot"></span></div>
        </div>
    </div>
</div>

<script>
var root__ID__ = document.getElementById("__ID__");
var fsBtnText__ID__ = document.getElementById("__ID__-fs-text");
function changeTechSlide__ID__(d) {{ let s = root__ID__.querySelectorAll('.tech-slide'); let dots = root__ID__.querySelectorAll('#tech-dots-__ID__ .dot'); let c = Array.from(s).findIndex(el => el.classList.contains('active')); s[c].classList.remove('active'); dots[c].classList.remove('active'); let next = (c + d + s.length) % s.length; s[next].classList.add('active'); dots[next].classList.add('active'); }}
function changeDataSlide__ID__(d) {{ let s = root__ID__.querySelectorAll('.data-slide'); let dots = root__ID__.querySelectorAll('#data-dots-__ID__ .dot'); let c = Array.from(s).findIndex(el => el.classList.contains('active')); s[c].classList.remove('active'); dots[c].classList.remove('active'); let next = (c + d + s.length) % s.length; s[next].classList.add('active'); dots[next].classList.add('active'); }}
function changeGraphSlide__ID__(d) {{ let s = root__ID__.querySelectorAll('.graph-slide-item'); let dots = root__ID__.querySelectorAll('#graph-dots-__ID__ .dot'); let c = Array.from(s).findIndex(el => el.classList.contains('active')); s[c].classList.remove('active'); dots[c].classList.remove('active'); let next = (c + d + s.length) % s.length; s[next].classList.add('active'); dots[next].classList.add('active'); }}
const galeria = ["https://sakwinetravel.com/wp-content/uploads/2018/07/tm0003-bodegas-uco2.jpg.webp", "https://dynamic-media-cdn.tripadvisor.com/media/photo-o/06/f7/6b/14/bodegas-caro.jpg?w=1200&h=-1&s=1", "https://www.amerian.com/wp-content/uploads/2025/11/zaca.jpg"];
let shuffled = galeria.sort(() => 0.5 - Math.random()).slice(0, 3);
const introCar = document.getElementById("__ID__-intro-carousel"); const mainCar  = document.getElementById("__ID__-main-carousel");
shuffled.forEach((src, idx) => {{ let i1 = document.createElement('img'); i1.src = src; if(idx === 0) i1.classList.add('active'); introCar.appendChild(i1); let i2 = document.createElement('img'); i2.src = src; if(idx === 0) i2.classList.add('active'); mainCar.appendChild(i2); }});
setInterval(() => {{ let s = introCar.querySelectorAll('img'); let c = Array.from(s).findIndex(el => el.classList.contains('active')); if(c >= 0) {{ s[c].classList.remove('active'); s[(c+1)%s.length].classList.add('active'); }} }}, 2500);
setInterval(() => {{ let s = mainCar.querySelectorAll('img'); let c = Array.from(s).findIndex(el => el.classList.contains('active')); if(c >= 0) {{ s[c].classList.remove('active'); s[(c+1)%s.length].classList.add('active'); }} }}, 4500);
function checkFullscreen() {{ if(fsBtnText__ID__) {{ fsBtnText__ID__.innerText = (document.fullscreenElement || window.innerHeight === screen.height) ? "Minimizar pantalla" : "Maximizar pantalla"; }} }}
document.addEventListener("fullscreenchange", checkFullscreen); window.addEventListener("resize", checkFullscreen);
function iniciarPresentacion__ID__() {{ let btnBox = root__ID__.querySelector('#__ID__-btnBox'); btnBox.style.opacity = 0; setTimeout(() => btnBox.style.display = 'none', 400); let content = root__ID__.querySelector('#__ID__-introContent'); content.style.display = 'flex'; let members = root__ID__.querySelectorAll('#__ID__-intro .member'); members.forEach((m, i) => {{ setTimeout(() => {{ m.style.opacity = '1'; m.style.transform = 'translateY(0)'; }}, i * 500 + 200); }}); setTimeout(() => {{ let stack = root__ID__.querySelector('#__ID__-memberStack'); stack.classList.add('crushed'); setTimeout(() => {{ let trubo = root__ID__.querySelector('#__ID__-trubo-presenta'); trubo.style.display = 'block'; setTimeout(() => {{ trubo.style.opacity = 1; }}, 50); setTimeout(() => {{ trubo.style.opacity = 0; setTimeout(() => {{ trubo.style.display = 'none'; let titleDiv = root__ID__.querySelector('#__ID__-introTitle'); titleDiv.style.display = 'block'; setTimeout(() => {{ titleDiv.classList.add('show'); }}, 50); setTimeout(() => {{ let intro = root__ID__.querySelector('#__ID__-intro'); let mainContent = root__ID__.querySelector('#__ID__-main'); intro.style.opacity = 0; mainContent.style.display = 'block'; setTimeout(() => {{ mainContent.style.opacity = 1; intro.style.display = 'none'; }}, 50); }}, 3500); }}, 1000); }}, 2500); }}, 500); }}, members.length * 500 + 1500); }}
var tabs__ID__ = root__ID__.querySelector(".tabs").querySelectorAll("button"); var contents__ID__ = root__ID__.querySelectorAll(".tab-content");
function openTab__ID__(index) {{ tabs__ID__.forEach((btn, i) => {{ if(i === index) btn.classList.add("on"); else btn.classList.remove("on"); }}); contents__ID__.forEach((c, i) => {{ if(i === index) c.classList.add("active"); else c.classList.remove("active"); }}); }}
document.getElementById("__ID__-fs").onclick = function() {{ var ok = function() {{ var clonedNode = root__ID__.cloneNode(true); var introClon = clonedNode.querySelector('#__ID__-intro'); if(root__ID__.querySelector('#__ID__-intro').style.display === 'none') {{ introClon.style.display = 'none'; }} var styleOverride = document.createElement('style'); styleOverride.innerHTML = '.gx {{ height: 100vh !important; width: 100vw !important; max-width: none !important; border-radius: 0 !important; }}'; var container = document.createElement('div'); container.appendChild(styleOverride); container.appendChild(clonedNode); var extraScript = "<script>function changeTechSlide__ID__(d){{ let s=document.querySelectorAll('.tech-slide'); let dots=document.querySelectorAll('#tech-dots-__ID__ .dot'); let c=Array.from(s).findIndex(el=>el.classList.contains('active')); s[c].classList.remove('active'); dots[c].classList.remove('active'); let next=(c+d+s.length)%s.length; s[next].classList.add('active'); dots[next].classList.add('active'); }}function changeDataSlide__ID__(d){{ let s=document.querySelectorAll('.data-slide'); let dots=document.querySelectorAll('#data-dots-__ID__ .dot'); let c=Array.from(s).findIndex(el=>el.classList.contains('active')); s[c].classList.remove('active'); dots[c].classList.remove('active'); let next=(c+d+s.length)%s.length; s[next].classList.add('active'); dots[next].classList.add('active'); }}function changeGraphSlide__ID__(d){{ let s=document.querySelectorAll('.graph-slide-item'); let dots=document.querySelectorAll('#graph-dots-__ID__ .dot'); let c=Array.from(s).findIndex(el=>el.classList.contains('active')); s[c].classList.remove('active'); dots[c].classList.remove('active'); let next=(c+d+s.length)%s.length; s[next].classList.add('active'); dots[next].classList.add('active'); }}document.getElementById('__ID__-fs').onclick=function(){{ if(document.fullscreenElement){{document.exitFullscreen();}}else{{document.documentElement.requestFullscreen();}} }};document.addEventListener('fullscreenchange',function(){{ var t=document.getElementById('__ID__-fs-text'); if(t){{t.innerText=document.fullscreenElement?'Minimizar pantalla':'Maximizar pantalla';}} }});function openTab__ID__(i){{ document.querySelectorAll('.tabs button').forEach((b,j)=>{{if(j===i)b.classList.add('on');else b.classList.remove('on');}}); document.querySelectorAll('.tab-content').forEach((c,j)=>{{if(j===i)c.classList.add('active');else c.classList.remove('active');}}); }}<\\/script>"; var b = new Blob(["<!doctype html><meta charset='utf-8'><body style='margin:0;padding:0;background:#fcf9f2;overflow-x:hidden;'>" + container.innerHTML + extraScript + "</body>"], {{type:"text/html"}}); var w = window.open(URL.createObjectURL(b), "_blank"); if(!w) alert("Permite las ventanas emergentes (pop-ups)."); }}; if(document.fullscreenElement) {{ document.exitFullscreen(); }} else {{ try {{ var r = root__ID__.requestFullscreen && root__ID__.requestFullscreen(); if(r && r.catch) r.catch(ok); else if(!r) ok(); }} catch(e) {{ ok(); }} }} }};
</script>
</div>
</body>
</html>
'''

_pagina_final = _html_template.replace('__ID__', 'gx' + uuid.uuid4().hex[:6])

archivo_exportar = "presentacion_data_winery.html"
with open(archivo_exportar, "w", encoding="utf-8") as file:
    file.write(_pagina_final)

print("✅ ¡Terminado! La presentación se guardó como 'presentacion_data_winery.html'.")
print("Puedes descargar el archivo desde la barra lateral izquierda de Colab (ícono de carpeta).")

1/5 - Descargando y descomprimiendo el dataset desde tu GitHub...
2/5 - Generando mapas de Europa...
3/5 - Generando mapa de Estados Unidos...
4/5 - Generando gráficos analíticos...
5/5 - Empaquetando todo en el archivo HTML (esto puede tardar unos segundos)...
✅ ¡Terminado! La presentación se guardó como 'presentacion_data_winery.html'.
Puedes descargar el archivo desde la barra lateral izquierda de Colab (ícono de carpeta).
